## Setup and Imports

In [ ]:
# Standard library imports
import sys
import warnings
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import numpy as np

# Data manipulation
import pandas as pd
import seaborn as sns

# Statistical analysis

# Configuration
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# Set display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

# Add project modules to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

print("Libraries imported successfully")
print(f"Project root: {project_root}")

## 1. Data Loading and Overview

In [ ]:
# Load synthetic data (actual path in your project)
data_path = project_root / "data" / "input" / "train" / "azure_ad_train.jsonl"

if not data_path.exists():
    print(f"Data file not found: {data_path}")
    print("\nPlease generate synthetic data first:")
    print("python scripts/utils/generate_azure_ad_data.py \\")
    print("    --output data/input/train/azure_ad_train.jsonl \\")
    print("    --num-events 10000 \\")
    print("    --num-users 50")
else:
    df = pd.read_json(data_path, lines=True)
    print(f"Data loaded: {len(df):,} events")
    print(f"Columns: {len(df.columns)}")
    print(f"\nDataset shape: {df.shape}")

In [ ]:
# Display first few rows
print("Sample Data:")
df.head()

In [ ]:
# Data types and basic info
print("Data Types and Non-Null Counts:")
df.info()

In [ ]:
# Convert timestamp to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Extract temporal features
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["dayofweek"] = df["timestamp"].dt.dayofweek
df["day_name"] = df["timestamp"].dt.day_name()

print("Temporal features extracted")

In [ ]:
# Basic statistics
print("Dataset Summary:")
print(f"  Total events: {len(df):,}")
print(f"  Unique users: {df['username'].nunique()}")
print(f"  Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"  Duration: {(df['timestamp'].max() - df['timestamp'].min()).days} days")

## 2. Feature Distributions

In [ ]:
# Identify numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")

In [ ]:
# Statistical summary of numeric features
print("Numeric Feature Statistics:")
df[numeric_cols].describe()

In [ ]:
# Distribution plots for key behavioral features
behavioral_features = ["logcount", "locincrement", "appincrement"]
available_features = [f for f in behavioral_features if f in df.columns]

if available_features:
    fig, axes = plt.subplots(1, len(available_features), figsize=(15, 4))
    if len(available_features) == 1:
        axes = [axes]

    for idx, feature in enumerate(available_features):
        axes[idx].hist(df[feature], bins=50, edgecolor="black", alpha=0.7)
        axes[idx].set_xlabel(feature, fontsize=12)
        axes[idx].set_ylabel("Frequency", fontsize=12)
        axes[idx].set_title(f"Distribution of {feature}", fontsize=14)
        axes[idx].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("Note: Behavioral features will be created during preprocessing")
    print("These include: logcount, locincrement, appincrement")

In [ ]:
# Box plots for numeric features
if numeric_cols:
    n_cols = min(4, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = axes.flatten() if n_rows > 1 or n_cols > 1 else [axes]

    for idx, col in enumerate(numeric_cols):
        if idx < len(axes):
            axes[idx].boxplot(df[col].dropna(), vert=True)
            axes[idx].set_ylabel(col, fontsize=12)
            axes[idx].set_title(f"Box Plot: {col}", fontsize=14)
            axes[idx].grid(True, alpha=0.3)

    # Hide unused subplots
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()

## 3. Temporal Pattern Analysis

In [ ]:
# Events per day
daily_counts = df.groupby("date").size()

plt.figure(figsize=(15, 5))
plt.plot(daily_counts.index, daily_counts.values, marker="o", linewidth=2)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Number of Events", fontsize=12)
plt.title("Daily Event Volume", fontsize=14, fontweight="bold")
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Average daily events: {daily_counts.mean():.0f}")
print(f"Min daily events: {daily_counts.min()}")
print(f"Max daily events: {daily_counts.max()}")

In [ ]:
# Hourly pattern analysis
hourly_counts = df.groupby("hour").size()

plt.figure(figsize=(15, 5))
plt.bar(hourly_counts.index, hourly_counts.values, edgecolor="black", alpha=0.7)
plt.xlabel("Hour of Day", fontsize=12)
plt.ylabel("Number of Events", fontsize=12)
plt.title("Hourly Event Distribution", fontsize=14, fontweight="bold")
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("\nBusiest hours:")
print(hourly_counts.nlargest(5))

In [ ]:
# Day of week pattern
dow_counts = (
    df.groupby("day_name")
    .size()
    .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
)

plt.figure(figsize=(12, 5))
plt.bar(range(7), dow_counts.values, tick_label=dow_counts.index, edgecolor="black", alpha=0.7)
plt.xlabel("Day of Week", fontsize=12)
plt.ylabel("Number of Events", fontsize=12)
plt.title("Weekly Event Distribution", fontsize=14, fontweight="bold")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("\nEvents by day of week:")
print(dow_counts)

In [ ]:
# Heatmap: Hour of day vs Day of week
heatmap_data = df.groupby(["day_name", "hour"]).size().unstack(fill_value=0)
heatmap_data = heatmap_data.reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])

plt.figure(figsize=(16, 6))
sns.heatmap(heatmap_data, cmap="YlOrRd", annot=False, fmt="d", cbar_kws={"label": "Event Count"})
plt.xlabel("Hour of Day", fontsize=12)
plt.ylabel("Day of Week", fontsize=12)
plt.title("Event Heatmap: Day of Week vs Hour of Day", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. User Behavior Profiling

In [ ]:
# User statistics
user_stats = (
    df.groupby("username")
    .agg(
        {
            "timestamp": ["count", "min", "max"],
        }
    )
    .round(2)
)

user_stats.columns = ["event_count", "first_event", "last_event"]
user_stats["duration_days"] = (user_stats["last_event"] - user_stats["first_event"]).dt.days
user_stats["events_per_day"] = user_stats["event_count"] / (user_stats["duration_days"] + 1)

print("User Activity Statistics:")
print(user_stats.describe())

In [ ]:
# Distribution of events per user
events_per_user = df["username"].value_counts()

plt.figure(figsize=(15, 5))
plt.hist(events_per_user.values, bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Number of Events", fontsize=12)
plt.ylabel("Number of Users", fontsize=12)
plt.title("Distribution of Events Per User", fontsize=14, fontweight="bold")
plt.axvline(
    events_per_user.mean(), color="red", linestyle="--", linewidth=2, label=f"Mean: {events_per_user.mean():.0f}"
)
plt.axvline(
    events_per_user.median(),
    color="green",
    linestyle="--",
    linewidth=2,
    label=f"Median: {events_per_user.median():.0f}",
)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTotal users: {len(events_per_user)}")
print(f"Average events per user: {events_per_user.mean():.2f}")
print(f"Median events per user: {events_per_user.median():.0f}")
print(f"Min events: {events_per_user.min()}")
print(f"Max events: {events_per_user.max()}")

In [ ]:
# Top 20 most active users
top_users = events_per_user.head(20)

plt.figure(figsize=(15, 6))
plt.barh(range(len(top_users)), top_users.values, edgecolor="black", alpha=0.7)
plt.yticks(range(len(top_users)), top_users.index)
plt.xlabel("Number of Events", fontsize=12)
plt.ylabel("Username", fontsize=12)
plt.title("Top 20 Most Active Users", fontsize=14, fontweight="bold")
plt.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

In [ ]:
# User activity over time (sample of users)
sample_users = events_per_user.head(10).index

plt.figure(figsize=(15, 8))
for user in sample_users:
    user_data = df[df["username"] == user].groupby("date").size()
    plt.plot(user_data.index, user_data.values, marker="o", label=user, linewidth=2, markersize=4)

plt.xlabel("Date", fontsize=12)
plt.ylabel("Number of Events", fontsize=12)
plt.title("User Activity Over Time (Top 10 Users)", fontsize=14, fontweight="bold")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=10)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Categorical Feature Analysis

In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in ["timestamp", "date", "day_name"]]

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Cardinality analysis
print("Categorical Feature Cardinality:")
for col in categorical_cols[:10]:  # First 10 categorical columns
    unique_count = df[col].nunique()
    print(f"  {col}: {unique_count} unique values")

In [ ]:
# Top values for key categorical features
key_categorical = ["appDisplayName", "clientAppUsed", "city", "country_region"]
available_categorical = [c for c in key_categorical if c in df.columns]

if available_categorical:
    for col in available_categorical:
        print(f"\nTop 10 values for '{col}':")
        value_counts = df[col].value_counts().head(10)
        for val, count in value_counts.items():
            pct = (count / len(df)) * 100
            print(f"  {val}: {count} ({pct:.2f}%)")

In [ ]:
# Visualize top applications
if "appDisplayName" in df.columns:
    top_apps = df["appDisplayName"].value_counts().head(15)

    plt.figure(figsize=(15, 6))
    plt.barh(range(len(top_apps)), top_apps.values, edgecolor="black", alpha=0.7)
    plt.yticks(range(len(top_apps)), top_apps.index)
    plt.xlabel("Number of Events", fontsize=12)
    plt.ylabel("Application", fontsize=12)
    plt.title("Top 15 Applications by Usage", fontsize=14, fontweight="bold")
    plt.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    plt.show()

In [ ]:
# Geographic distribution
if "country_region" in df.columns:
    country_counts = df["country_region"].value_counts().head(15)

    plt.figure(figsize=(15, 6))
    plt.bar(range(len(country_counts)), country_counts.values, edgecolor="black", alpha=0.7)
    plt.xticks(range(len(country_counts)), country_counts.index, rotation=45, ha="right")
    plt.xlabel("Country", fontsize=12)
    plt.ylabel("Number of Events", fontsize=12)
    plt.title("Top 15 Countries by Event Count", fontsize=14, fontweight="bold")
    plt.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()

## 6. Missing Value Analysis

In [ ]:
# Calculate missing values
missing_values = df.isnull().sum()
missing_percentages = (missing_values / len(df)) * 100

missing_df = pd.DataFrame(
    {
        "Column": missing_values.index,
        "Missing Count": missing_values.values,
        "Missing Percentage": missing_percentages.values,
    }
).sort_values("Missing Count", ascending=False)

# Show only columns with missing values
missing_df = missing_df[missing_df["Missing Count"] > 0]

if len(missing_df) > 0:
    print("Columns with Missing Values:")
    print(missing_df.to_string(index=False))

    # Visualize missing values
    if len(missing_df) > 0:
        plt.figure(figsize=(12, max(6, len(missing_df) * 0.4)))
        plt.barh(range(len(missing_df)), missing_df["Missing Percentage"].values, edgecolor="black", alpha=0.7)
        plt.yticks(range(len(missing_df)), missing_df["Column"].values)
        plt.xlabel("Missing Percentage (%)", fontsize=12)
        plt.ylabel("Column", fontsize=12)
        plt.title("Missing Value Percentages by Column", fontsize=14, fontweight="bold")
        plt.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        plt.show()
else:
    print("No missing values found in the dataset.")

## 7. Feature Correlations

In [ ]:
# Correlation matrix for numeric features
if len(numeric_cols) > 1:
    correlation_matrix = df[numeric_cols].corr()

    plt.figure(figsize=(12, 10))
    sns.heatmap(
        correlation_matrix,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        square=True,
        linewidths=1,
        cbar_kws={"shrink": 0.8},
    )
    plt.title("Feature Correlation Heatmap", fontsize=14, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.show()

    # Print highly correlated pairs
    print("\nHighly Correlated Feature Pairs (|correlation| > 0.7):")
    for i in range(len(correlation_matrix.columns)):
        for j in range(i + 1, len(correlation_matrix.columns)):
            corr_val = correlation_matrix.iloc[i, j]
            if abs(corr_val) > 0.7:
                col1 = correlation_matrix.columns[i]
                col2 = correlation_matrix.columns[j]
                print(f"  {col1} <-> {col2}: {corr_val:.3f}")
else:
    print("Not enough numeric features for correlation analysis")

## Summary and Key Insights

In [ ]:
print("=" * 80)
print("EXPLORATORY DATA ANALYSIS SUMMARY")
print("=" * 80)

print("\n1. DATASET OVERVIEW")
print(f"   Total events: {len(df):,}")
print(f"   Total users: {df['username'].nunique()}")
print(f"   Date range: {(df['timestamp'].max() - df['timestamp'].min()).days} days")
print(f"   Features: {len(df.columns)}")

print("\n2. USER BEHAVIOR")
print(f"   Average events per user: {events_per_user.mean():.2f}")
print(f"   Median events per user: {events_per_user.median():.0f}")
print(f"   Users with >300 events: {(events_per_user > 300).sum()}")

print("\n3. TEMPORAL PATTERNS")
print(f"   Peak hour: {hourly_counts.idxmax()}:00 ({hourly_counts.max()} events)")
print(f"   Busiest day: {dow_counts.idxmax()} ({dow_counts.max()} events)")
print(f"   Average daily events: {daily_counts.mean():.0f}")

print("\n4. DATA QUALITY")
if len(missing_df) > 0:
    print(f"   Columns with missing values: {len(missing_df)}")
    print(f"   Max missing percentage: {missing_df['Missing Percentage'].max():.2f}%")
else:
    print("   No missing values detected")

print("\n5. CATEGORICAL FEATURES")
if "appDisplayName" in df.columns:
    print(f"   Unique applications: {df['appDisplayName'].nunique()}")
if "country_region" in df.columns:
    print(f"   Unique countries: {df['country_region'].nunique()}")
if "city" in df.columns:
    print(f"   Unique cities: {df['city'].nunique()}")

print("\n6. RECOMMENDATIONS FOR MODELING")
print("   - Sufficient data for per-user modeling (300+ events)")
print("   - Strong temporal patterns suggest time-based features are valuable")
print("   - Geographic diversity indicates location-based anomalies possible")
print("   - Application usage patterns show normal behavior baselines")

print("\n" + "=" * 80)

## Next Steps

Based on this exploratory analysis:

1. **Preprocessing:** Apply DFPPreprocessing module to extract behavioral features (logcount, locincrement, appincrement)
2. **Rolling Window:** Use RollingWindow module for temporal aggregation
3. **Training:** Train per-user AutoEncoder models on users with sufficient data
4. **Inference:** Apply trained models to detect anomalous behavior

See other notebooks:
- `reconstruction_error_analysis.ipynb` - Analyze model performance
- `rolling_window_visualization.ipynb` - Visualize window aggregations
- `model_comparison.ipynb` - Compare user-specific vs generic models